# 02b — Run the Intake Agent

Inference layer: runs legal intake submissions through the agent (built from `agent_lib.py`), persists structured intake profiles to the Delta-backed intake state store, and populates the HITL attorney-review queue.

### Key Functions

- Process incoming legal intake narratives through the ReAct agent
- Execute semantic retrieval, conflict checking, and case routing workflows
- Persist structured intake profiles to the Delta-backed state store
- Queue completed intakes for attorney review (`reviewed = false`)
- Demonstrate the system's three core behaviors:
  - Case routing
  - Conflict checking
  - Graceful out-of-scope rejection

### Notes
**Switch this notebook to Serverless CPU compute to run all. This Python notebook is attached to a SQL warehouse, which only supports SQL and Markdown cells.

This notebook preserves the original `provision_text` widget and `run_react_agent()` function signature to maintain compatibility with downstream evaluation and benchmarking workflows.

In [0]:
# Configure Widgets
dbutils.widgets.text("catalog", "workspace", "Unity Catalog name")
dbutils.widgets.text("schema", "default", "Schema")
dbutils.widgets.text("vs_endpoint", "lexpath_vs_endpoint", "Vector Search Endpoint")
dbutils.widgets.text("llm_endpoint", "anthropic-claude-sonnet-4-6", "LLM Serving Endpoint")
dbutils.widgets.text("provision_text", "", "Target Provision Text Payload")  # kept from stub

In [0]:
# Install LangChain/Databricks/Vector Search/MLflow Stack
%pip install --upgrade 'langchain>=1.3.0' langgraph databricks-langchain databricks-vectorsearch mlflow

  Using cached langchain-1.3.9-py3-none-any.whl.metadata (5.8 kB)
  Using cached langgraph-1.2.5-py3-none-any.whl.metadata (8.0 kB)
  Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_sdk-0.4.2-py3-none-any.whl.metadata (3.6 kB)
  Using cached websockets-15.0.1-cp312-cp312-manylinux_2_17_aarch64.manylinux2014_aarch64.whl.metadata (6.8 kB)
INFO: pip is looking at multiple versions of unitycatalog-langchain to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of unitycatalog-langchain[databricks] to determine which version is compatible with other requirements. This could take a while.
  Attempting uninstall: websockets
    Found existing installation: websockets 16.0
    Uninstalling websockets-16.0:
      Successfully uninstalled websockets-16.0
  Attempting uninstall: langgraph-sdk
    Found existing installation: langgraph-sdk 0.3.15
    Uninstalling langgra

In [0]:
# Restart Python
dbutils.library.restartPython()

In [0]:
# Import Libraries, Configure Agent, and Setup MLflow
import sys, os, json
import importlib
sys.path.append(os.getcwd())
 
import mlflow
import agent_lib
importlib.reload(agent_lib)  # Reload to pick up any file changes
from pyspark.sql import functions as F
 
agent_lib.configure(
    catalog=dbutils.widgets.get("catalog"),
    schema=dbutils.widgets.get("schema"),
    vs_endpoint=dbutils.widgets.get("vs_endpoint"),
)
mlflow.langchain.autolog()
 
LLM_ENDPOINT = dbutils.widgets.get("llm_endpoint")
INTAKE_TABLE = f"{agent_lib.CATALOG}.{agent_lib.SCHEMA}.lexpath_intake_profiles"
 
executor = agent_lib.build_agent(LLM_ENDPOINT)
print(f"Agent ready on {LLM_ENDPOINT}")

agent_lib configured — index workspace.default.ledgar_provisions_index, 100 routing labels
Agent ready on anthropic-claude-sonnet-4-6


In [0]:
# Inference layer + Delta-backed intake state store
def save_intake_profile(raw_text: str, profile: dict) -> None:
    """Persist the structured intake profile for HITL attorney review."""
    row = spark.createDataFrame(
        [(raw_text, json.dumps(profile), profile.get("status", ""),
          profile.get("predicted_category", ""), profile.get("practice_area", ""),
          profile.get("conflict_status", ""))],
        schema=("raw_intake_text string, profile_json string, status string, "
                "predicted_category string, practice_area string, conflict_status string"),
    ).withColumn("created_at", F.current_timestamp()).withColumn("reviewed", F.lit(False))
    row.write.format("delta").mode("append").saveAsTable(INTAKE_TABLE)
 
def run_react_agent(text_payload: str) -> str:
    """
    Core ReAct Agent processing logic. (Same signature as the original stub.)
    Runs the full tool loop, persists the intake profile, and returns the
    predicted LEDGAR category label for downstream evaluation.
    """
    if not text_payload.strip():
        return "Unknown"
 
    result = executor.invoke({"input": text_payload})
    profile = agent_lib.extract_json(result.get("output", ""))
    if not profile:
        return "Unknown"
 
    save_intake_profile(text_payload, profile)
    return profile.get("predicted_category") or profile.get("status", "Unknown")

In [0]:
# Inference layer + Delta-backed intake state store

def save_intake_profile(raw_text: str, profile: dict) -> None:
    """Persist the structured intake profile for HITL attorney review."""
    row = spark.createDataFrame(
        [(raw_text, json.dumps(profile), profile.get("status", ""),
          profile.get("predicted_category", ""), profile.get("practice_area", ""),
          profile.get("conflict_status", ""))],
        schema=("raw_intake_text string, profile_json string, status string, "
                "predicted_category string, practice_area string, conflict_status string"),
    ).withColumn("created_at", F.current_timestamp()).withColumn("reviewed", F.lit(False))
    row.write.format("delta").mode("append").saveAsTable(INTAKE_TABLE)
 
def run_react_agent(text_payload: str) -> str:
    """
    Core ReAct Agent processing logic. 
    Runs the full tool loop, persists the intake profile, and returns the
    predicted LEDGAR category label for downstream evaluation.
    """
    if not text_payload.strip():
        return "Unknown"
 
    result = executor.invoke({"input": text_payload})
    profile = agent_lib.extract_json(result.get("output", ""))
    if not profile:
        return "Unknown"
 
    save_intake_profile(text_payload, profile)
    return profile.get("predicted_category") or profile.get("status", "Unknown")

In [0]:
# Standalone widget-driven test
input_text = dbutils.widgets.get("provision_text")
if input_text:
    route = run_react_agent(input_text)
    print(f"🤖 Agent Predicted Route: {route}")

In [0]:
# Demo scenarios: normal, conflict, out-of-scope
demo_intakes = [
    # 1. Normal intake — should retrieve, classify, route; no parties → NOT_RUN
    "My business partner and I signed an agreement that says disputes go to arbitration, "
    "but now they filed a lawsuit in court instead. I want to enforce the arbitration clause.",
 
    # 2. Conflict scenario — names an existing client from lexpath_conflicts (01c)
    "I want to sue Atlas Manufacturing. I was injured by one of their forklifts at a "
    "warehouse in March and they refuse to cover my medical bills. My name is Paul Vance.",
 
    # 3. Out-of-scope — should be gracefully rejected without tool calls
    "Can you just tell me whether I'd win if I represented myself? Give me your legal "
    "opinion on my chances.",
]
 
for i, intake in enumerate(demo_intakes, 1):
    print(f"\n{'='*80}\nSCENARIO {i}: {intake[:90]}...\n")
    label = run_react_agent(intake)
    print(f"\n🤖 Returned label/status: {label}")


SCENARIO 1: My business partner and I signed an agreement that says disputes go to arbitration, but no...

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

🤖 Returned label/status: Arbitration

SCENARIO 2: I want to sue Atlas Manufacturing. I was injured by one of their forklifts at a warehouse ...

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

🤖 Returned label/status: Indemnity

SCENARIO 3: Can you just tell me whether I'd win if I represented myself? Give me your legal opinion o...


🤖 Returned label/status: REJECTED_OUT_OF_SCOPE


[Trace(trace_id=tr-5d47a7e1c85eac5e29d98bd57e0d4d3c), Trace(trace_id=tr-e3e21f6ce268aa7a7bf6dca0e7656ad4), Trace(trace_id=tr-04853ccb200730ff8810aeab3f3ac4ba)]

In [0]:
# Inspect the Delta-backed intake state store (HITL review queue)
display(
    spark.table(INTAKE_TABLE)
         .orderBy(F.desc("created_at"))
         .limit(10)
)

raw_intake_text,profile_json,status,predicted_category,practice_area,conflict_status,created_at,reviewed
Can you just tell me whether I'd win if I represented myself? Give me your legal opinion on my chances.,"{""status"": ""REJECTED_OUT_OF_SCOPE"", ""issue_summary"": ""The prospective client is requesting a legal opinion on their chances of success in self-representation. This agent is a client-intake system only and is not able to provide legal advice, legal opinions, or assessments of litigation outcomes. A licensed attorney must be consulted for any such guidance."", ""predicted_category"": """", ""practice_area"": """", ""parties"": [], ""conflict_status"": ""NOT_RUN"", ""conflict_matches"": [], ""clarifying_questions"": [], ""routing_rationale"": """"}",REJECTED_OUT_OF_SCOPE,,,NOT_RUN,2026-06-13T05:40:32.697Z,false
I want to sue Atlas Manufacturing. I was injured by one of their forklifts at a warehouse in March and they refuse to cover my medical bills. My name is Paul Vance.,"{""status"": ""READY_FOR_REVIEW"", ""issue_summary"": ""Prospective client Paul Vance reports being injured by a forklift operated or manufactured by Atlas Manufacturing at a warehouse in March. Atlas Manufacturing has declined to cover his medical expenses arising from the incident. Mr. Vance is seeking to pursue legal action against Atlas Manufacturing for damages."", ""predicted_category"": ""Indemnity"", ""practice_area"": ""Corporate"", ""parties"": [""Paul Vance"", ""Atlas Manufacturing""], ""conflict_status"": ""CONFLICT_FLAG"", ""conflict_matches"": [{""matter_id"": ""M-1005"", ""client"": ""Tom Garrety"", ""opposing_party"": ""Atlas Manufacturing"", ""status"": ""active""}, {""matter_id"": ""M-1006"", ""client"": ""Atlas Manufacturing"", ""opposing_party"": ""Union Local 482"", ""status"": ""closed""}], ""clarifying_questions"": [], ""routing_rationale"": ""Retrieved provisions most closely matching this matter involve indemnity obligations \u2014 specifically, clauses addressing liability for personal injury arising from product defects and the duty of parties to hold harmless and cover damages, costs, and claims. This aligns with Mr. Vance's claim that Atlas Manufacturing should bear responsibility for his medical expenses following the forklift incident.""}",READY_FOR_REVIEW,Indemnity,Corporate,CONFLICT_FLAG,2026-06-13T05:40:25.839Z,false
"My business partner and I signed an agreement that says disputes go to arbitration, but now they filed a lawsuit in court instead. I want to enforce the arbitration clause.","{""status"": ""READY_FOR_REVIEW"", ""issue_summary"": ""The prospective client entered into a business agreement containing an arbitration clause requiring disputes to be resolved through arbitration. Their business partner has since filed a lawsuit in court in apparent breach of that clause. The client seeks to enforce the arbitration agreement and compel the matter out of court."", ""predicted_category"": ""Arbitration"", ""practice_area"": ""Litigation"", ""parties"": [], ""conflict_status"": ""NOT_RUN"", ""conflict_matches"": [], ""clarifying_questions"": [], ""routing_rationale"": ""Multiple retrieved LEDGAR provisions closely match this matter, consistently categorizing exclusive arbitration clauses \u2014 where disputes 'shall be settled exclusively by arbitration' \u2014 under the 'Arbitration' label. The client's goal of enforcing such a clause against a party who filed in court aligns directly with these provisions.""}",READY_FOR_REVIEW,Arbitration,Litigation,NOT_RUN,2026-06-13T05:40:09.617Z,false
"My business partner and I signed an agreement that says disputes go to arbitration, but now they filed a lawsuit in court instead. I want to enforce the arbitration clause.","{""status"": ""READY_FOR_REVIEW"", ""issue_summary"": ""The prospective client entered into a business agreement containing an arbitration clause requiring disputes to be resolved through arbitration. Their business partner has since filed a lawsuit in 

## Summary

- **Shared core**: Tools, prompts, and agent construction are centralized in `agent_lib.py`, ensuring that notebooks 02a, 02b, and 03 remain consistent and cannot drift apart.
- **ReAct loop**: The model follows a reasoning cycle of **think → select tool → observe result → iterate**, with a maximum of 8 tool-calling iterations.
- **MLflow autologging** records every reasoning trace, tool input/output, and retrieved document. Detailed execution traces can be reviewed in the experiment's **Traces** tab.
- **Intake state store**: Each completed run appends a structured JSON profile to `lexpath_intake_profiles` with `reviewed = false`, creating a Delta-backed HITL attorney-review queue.
- **Stable interface**: `run_react_agent()` preserves the original contract of **text in → label out**, enabling seamless integration with downstream evaluation and benchmarking workflows.